<a href="https://colab.research.google.com/github/Gael199/Final_Projet/blob/master/Exercice_SPARK_PROJET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q pyspark

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("PySpark_Student_Exercises_Movies")
    .master("local[*]")
    .getOrCreate()
)

spark

In [2]:
!wget -q https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip -q ml-latest-small.zip

ratings = spark.read.csv("ml-latest-small/ratings.csv", header=True, inferSchema=True)
movies  = spark.read.csv("ml-latest-small/movies.csv", header=True, inferSchema=True)

In [3]:
# Full solution for the MovieLens exercises (PySpark)
# Paste into your notebook after the setup cell that creates `spark`
# and after you've loaded `ratings` and `movies` as in your notebook.

from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import IntegerType, FloatType
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import time

# Task 1 — Data Exploration

### 1.1 Inspecter les données

In [4]:
# Show first 10 rows
ratings.show(10, truncate=False)
movies.show(10, truncate=False)

# Count rows
print("Nombre de ratings:", ratings.count())
print("Nombre de movies:", movies.count())

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|1     |1      |4.0   |964982703|
|1     |3      |4.0   |964981247|
|1     |6      |4.0   |964982224|
|1     |47     |5.0   |964983815|
|1     |50     |5.0   |964982931|
|1     |70     |3.0   |964982400|
|1     |101    |5.0   |964980868|
|1     |110    |4.0   |964982176|
|1     |151    |5.0   |964984041|
|1     |157    |5.0   |964984100|
+------+-------+------+---------+
only showing top 10 rows
+-------+----------------------------------+-------------------------------------------+
|movieId|title                             |genres                                     |
+-------+----------------------------------+-------------------------------------------+
|1      |Toy Story (1995)                  |Adventure|Animation|Children|Comedy|Fantasy|
|2      |Jumanji (1995)                    |Adventure|Children|Fantasy                 |
|3      |Grumpier Old Men (1995)           |Comedy|Rom

### 1.2 Sélection & renommage

In [6]:
from pyspark.sql.functions import col

In [7]:
# select only userId, movieId, rating; rename movieId -> film_id; cast rating to float
ratings_sel = ratings.select(
    col("userId"),
    col("movieId").alias("film_id"),
    col("rating").cast("float")
)
ratings_sel.show(10)
ratings_sel.printSchema()

+------+-------+------+
|userId|film_id|rating|
+------+-------+------+
|     1|      1|   4.0|
|     1|      3|   4.0|
|     1|      6|   4.0|
|     1|     47|   5.0|
|     1|     50|   5.0|
|     1|     70|   3.0|
|     1|    101|   5.0|
|     1|    110|   4.0|
|     1|    151|   5.0|
|     1|    157|   5.0|
+------+-------+------+
only showing top 10 rows
root
 |-- userId: integer (nullable = true)
 |-- film_id: integer (nullable = true)
 |-- rating: float (nullable = true)



### 1.3 Filtrage

In [8]:
# All ratings by userId = 1
ratings.filter(col("userId") == 1).show(10)

# Ratings > 4.5
ratings.filter(col("rating") > 4.5).show(10)

# Ratings where movieId in (1,50,100)
ratings.filter(col("movieId").isin(1,50,100)).show(20)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
|     1|     70|   3.0|964982400|
|     1|    101|   5.0|964980868|
|     1|    110|   4.0|964982176|
|     1|    151|   5.0|964984041|
|     1|    157|   5.0|964984100|
+------+-------+------+---------+
only showing top 10 rows
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
|     1|    101|   5.0|964980868|
|     1|    151|   5.0|964984041|
|     1|    157|   5.0|964984100|
|     1|    163|   5.0|964983650|
|     1|    216|   5.0|964981208|
|     1|    231|   5.0|964981179|
|     1|    260|   5.0|964981680|
|     1|    333|   5.0|964981179|
+------+-------+------+---------+
only showing top 10 row

### 1.4 Tri & limit

In [9]:
# Top 20 highest ratings (rating desc, timestamp desc to break ties)
ratings.orderBy(col("rating").desc(), col("timestamp").desc()).show(20)

# 10 lowest ratings by user 600 (rating asc, timestamp asc)
ratings.filter(col("userId") == 600).orderBy(col("rating").asc(), col("timestamp").asc()).show(10)

+------+-------+------+----------+
|userId|movieId|rating| timestamp|
+------+-------+------+----------+
|   210| 177765|   5.0|1537632257|
|   305| 148671|   5.0|1537354985|
|   331|  57669|   5.0|1537235356|
|   331|  55820|   5.0|1537235354|
|   331|    608|   5.0|1537235353|
|   331|   1213|   5.0|1537235351|
|   331|  54997|   5.0|1537158433|
|   331|   3969|   5.0|1537158394|
|   331|  94864|   5.0|1537158375|
|   331|  61323|   5.0|1537158300|
|   331|  91658|   5.0|1537158263|
|   331|  88129|   5.0|1537158250|
|   331|  97921|   5.0|1537158216|
|   331|  56782|   5.0|1537158113|
|   331| 112552|   5.0|1537158085|
|   331| 115713|   5.0|1537158030|
|   331| 109374|   5.0|1537158021|
|   331|   3911|   5.0|1537158005|
|   331| 112556|   5.0|1537157990|
|   331|   3535|   5.0|1537157975|
+------+-------+------+----------+
only showing top 20 rows
+------+-------+------+----------+
|userId|movieId|rating| timestamp|
+------+-------+------+----------+
|   600|    762|   0.5|1237707

### 1.5 Colonnes dérivées

In [11]:
from pyspark.sql.functions import col, when, lit, log10

In [12]:
from pyspark.sql.functions import log10

ratings_derived = ratings.withColumn("rating_x2", col("rating") * 2) \
                         .withColumn("positive_rating", when(col("rating") >= 4.0, lit(1)).otherwise(lit(0))) \
                         .withColumn("log_rating", log10(col("rating") + lit(1)))
ratings_derived.select("userId","movieId","rating","rating_x2","positive_rating","log_rating").show(10)

+------+-------+------+---------+---------------+------------------+
|userId|movieId|rating|rating_x2|positive_rating|        log_rating|
+------+-------+------+---------+---------------+------------------+
|     1|      1|   4.0|      8.0|              1|0.6989700043360189|
|     1|      3|   4.0|      8.0|              1|0.6989700043360189|
|     1|      6|   4.0|      8.0|              1|0.6989700043360189|
|     1|     47|   5.0|     10.0|              1|0.7781512503836436|
|     1|     50|   5.0|     10.0|              1|0.7781512503836436|
|     1|     70|   3.0|      6.0|              0|0.6020599913279624|
|     1|    101|   5.0|     10.0|              1|0.7781512503836436|
|     1|    110|   4.0|      8.0|              1|0.6989700043360189|
|     1|    151|   5.0|     10.0|              1|0.7781512503836436|
|     1|    157|   5.0|     10.0|              1|0.7781512503836436|
+------+-------+------+---------+---------------+------------------+
only showing top 10 rows


### 1.6 Missing values (démo)

In [13]:
# Add fake null column
ratings_null = ratings.withColumn("fake_null", when(col("userId") % 10 == 0, None).otherwise(col("userId")))
ratings_null.select("userId","fake_null").show(15)

# Fill nulls
ratings_filled = ratings_null.na.fill({"fake_null": -1})
ratings_filled.select("userId","fake_null").show(10)

# Drop rows with any nulls (example)
ratings_null.dropna().show(5)

+------+---------+
|userId|fake_null|
+------+---------+
|     1|        1|
|     1|        1|
|     1|        1|
|     1|        1|
|     1|        1|
|     1|        1|
|     1|        1|
|     1|        1|
|     1|        1|
|     1|        1|
|     1|        1|
|     1|        1|
|     1|        1|
|     1|        1|
|     1|        1|
+------+---------+
only showing top 15 rows
+------+---------+
|userId|fake_null|
+------+---------+
|     1|        1|
|     1|        1|
|     1|        1|
|     1|        1|
|     1|        1|
|     1|        1|
|     1|        1|
|     1|        1|
|     1|        1|
|     1|        1|
+------+---------+
only showing top 10 rows
+------+-------+------+---------+---------+
|userId|movieId|rating|timestamp|fake_null|
+------+-------+------+---------+---------+
|     1|      1|   4.0|964982703|        1|
|     1|      3|   4.0|964981247|        1|
|     1|      6|   4.0|964982224|        1|
|     1|     47|   5.0|964983815|        1|
|     1|     50

### 1.7 Distinct & deduplication

In [14]:
# Unique users and movies
print("Unique users:", ratings.select("userId").distinct().count())
print("Unique movies rated:", ratings.select("movieId").distinct().count())

# Drop duplicate ratings on (userId,movieId)
ratings_dedup = ratings.dropDuplicates(["userId","movieId"])
print("Before dropDuplicates:", ratings.count())
print("After dropDuplicates:", ratings_dedup.count())

Unique users: 610
Unique movies rated: 9724
Before dropDuplicates: 100836
After dropDuplicates: 100836


# Task 2 — Aggregations & GroupBy

### 2.1 Agrégations simples

In [15]:
# min, max, avg rating
ratings.select(F.min("rating").alias("min_rating"),
               F.max("rating").alias("max_rating"),
               F.avg("rating").alias("avg_rating")).show()

# total number of ratings
print("Total ratings:", ratings.count())

# count ratings per movieId
ratings.groupBy("movieId").count().show(10)

+----------+----------+-----------------+
|min_rating|max_rating|       avg_rating|
+----------+----------+-----------------+
|       0.5|       5.0|3.501556983616962|
+----------+----------+-----------------+

Total ratings: 100836
+-------+-----+
|movieId|count|
+-------+-----+
|   1580|  165|
|   2366|   25|
|   3175|   75|
|   1088|   42|
|  32460|    4|
|  44022|   23|
|  96488|    4|
|   1238|    9|
|   1342|   11|
|   1591|   26|
+-------+-----+
only showing top 10 rows


### 2.2 GroupBy: moyenne / nombre par movie et par user

In [16]:
# average rating per movie, number of ratings per movie
movie_stats = ratings.groupBy("movieId").agg(
    F.avg("rating").alias("avg_rating"),
    F.count("*").alias("num_ratings")
)
movie_stats.orderBy(col("num_ratings").desc()).show(10)

# average rating per user, number of ratings per user
user_stats = ratings.groupBy("userId").agg(
    F.avg("rating").alias("avg_rating_user"),
    F.count("*").alias("num_ratings_user")
)
user_stats.orderBy(col("num_ratings_user").desc()).show(10)

+-------+-----------------+-----------+
|movieId|       avg_rating|num_ratings|
+-------+-----------------+-----------+
|    356|4.164133738601824|        329|
|    318|4.429022082018927|        317|
|    296|4.197068403908795|        307|
|    593|4.161290322580645|        279|
|   2571|4.192446043165468|        278|
|    260|4.231075697211155|        251|
|    480|             3.75|        238|
|    110|4.031645569620253|        237|
|    589|3.970982142857143|        224|
|    527|            4.225|        220|
+-------+-----------------+-----------+
only showing top 10 rows
+------+------------------+----------------+
|userId|   avg_rating_user|num_ratings_user|
+------+------------------+----------------+
|   414| 3.391957005189029|            2698|
|   599|2.6420500403551253|            2478|
|   474| 3.398956356736243|            2108|
|   448|2.8473712446351933|            1864|
|   274| 3.235884101040119|            1346|
|   610|3.6885560675883258|            1302|
|    68| 3

### 2.3 Top items

### Top 20 les plus évalués

In [17]:
top20_most_rated = movie_stats.orderBy(col("num_ratings").desc()).limit(20)
top20_most_rated.show(truncate=False)

+-------+------------------+-----------+
|movieId|avg_rating        |num_ratings|
+-------+------------------+-----------+
|356    |4.164133738601824 |329        |
|318    |4.429022082018927 |317        |
|296    |4.197068403908795 |307        |
|593    |4.161290322580645 |279        |
|2571   |4.192446043165468 |278        |
|260    |4.231075697211155 |251        |
|480    |3.75              |238        |
|110    |4.031645569620253 |237        |
|589    |3.970982142857143 |224        |
|527    |4.225             |220        |
|2959   |4.272935779816514 |218        |
|1      |3.9209302325581397|215        |
|1196   |4.2156398104265405|211        |
|50     |4.237745098039215 |204        |
|2858   |4.056372549019608 |204        |
|47     |3.9753694581280787|203        |
|780    |3.4455445544554455|202        |
|150    |3.845771144278607 |201        |
|1198   |4.2075            |200        |
|4993   |4.106060606060606 |198        |
+-------+------------------+-----------+



### Top 20 meilleurs films (min 50 ratings)

In [18]:
top20_best_min50 = movie_stats.filter(col("num_ratings") >= 50) \
                              .orderBy(col("avg_rating").desc(), col("num_ratings").desc()) \
                              .limit(20)
top20_best_min50.show(truncate=False)

+-------+------------------+-----------+
|movieId|avg_rating        |num_ratings|
+-------+------------------+-----------+
|318    |4.429022082018927 |317        |
|858    |4.2890625         |192        |
|2959   |4.272935779816514 |218        |
|1276   |4.271929824561403 |57         |
|750    |4.268041237113402 |97         |
|904    |4.261904761904762 |84         |
|1221   |4.25968992248062  |129        |
|48516  |4.252336448598131 |107        |
|1213   |4.25              |126        |
|912    |4.24              |100        |
|58559  |4.238255033557047 |149        |
|50     |4.237745098039215 |204        |
|1197   |4.232394366197183 |142        |
|260    |4.231075697211155 |251        |
|527    |4.225             |220        |
|1208   |4.219626168224299 |107        |
|2329   |4.217054263565892 |129        |
|1196   |4.2156398104265405|211        |
|1252   |4.211864406779661 |59         |
|1198   |4.2075            |200        |
+-------+------------------+-----------+



In [19]:
top20_best_min50.join(movies, top20_best_min50.movieId == movies.movieId) \
                .select("movieId","title","avg_rating","num_ratings") \
                .orderBy(col("avg_rating").desc(), col("num_ratings").desc()) \
                .show(20,truncate=False)

{"ts": "2025-12-09 22:11:03.925", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[AMBIGUOUS_REFERENCE] Reference `movieId` is ambiguous, could be: [`movieId`, `movieId`]. SQLSTATE: 42704", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor54.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "AMBIGUOUS_REFERENCE"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o280.select.\n: org.apache.spark.sql.AnalysisException: [AMBIGUOUS_REFERENCE] Reference `movieId` is ambiguous, could be: [`movieId`, `movieId`]. SQLSTATE: 42704\n\tat org.apache.spark.sql.errors.QueryCompilationErrors$.ambiguousReferenceError(QueryCompilationErrors.scala:2163)\n\tat org.apache.spark.sql.catalyst.expressions.package$AttributeSeq.resolve(package.scala:356)\n\tat org.apache.spark.sql.catalyst.plans.logical.LogicalPlan.resolveChildren(LogicalPlan.scala:164)\n\tat org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.$anon

AnalysisException: [AMBIGUOUS_REFERENCE] Reference `movieId` is ambiguous, could be: [`movieId`, `movieId`]. SQLSTATE: 42704

In [20]:
top20_best_min50.join(movies, "movieId") \
    .select("movieId","title","avg_rating","num_ratings") \
    .orderBy(col("avg_rating").desc(), col("num_ratings").desc()) \
    .show(20, truncate=False)

+-------+------------------------------------------------------------------------------+------------------+-----------+
|movieId|title                                                                         |avg_rating        |num_ratings|
+-------+------------------------------------------------------------------------------+------------------+-----------+
|318    |Shawshank Redemption, The (1994)                                              |4.429022082018927 |317        |
|858    |Godfather, The (1972)                                                         |4.2890625         |192        |
|2959   |Fight Club (1999)                                                             |4.272935779816514 |218        |
|1276   |Cool Hand Luke (1967)                                                         |4.271929824561403 |57         |
|750    |Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)   |4.268041237113402 |97         |
|904    |Rear Window (1954)             

### 2.4 Window functions

In [21]:
from pyspark.sql.window import Window

# rank users' ratings per movie by timestamp (earliest -> latest)
w_movie = Window.partitionBy("movieId").orderBy(col("timestamp").asc())

ranked = ratings.withColumn("rank_by_time", F.row_number().over(w_movie))
ranked.select("userId","movieId","rating","timestamp","rank_by_time").show(20)

# lag: previous rating for same movie by timestamp order
ranked_lag = ratings.withColumn("prev_rating_for_movie", F.lag("rating").over(w_movie))
ranked_lag.select("userId","movieId","rating","prev_rating_for_movie").show(20)

# average rating per movie using window (same as groupBy but as window)
w_movie2 = Window.partitionBy("movieId")
ratings.withColumn("avg_rating_movie_window", F.avg("rating").over(w_movie2)) \
       .select("movieId","rating","avg_rating_movie_window").show(20)

+------+-------+------+---------+------------+
|userId|movieId|rating|timestamp|rank_by_time|
+------+-------+------+---------+------------+
|   107|      1|   4.0|829322340|           1|
|   191|      1|   4.0|829759809|           2|
|    54|      1|   3.0|830247330|           3|
|   468|      1|   4.0|831400444|           4|
|   353|      1|   5.0|831939685|           5|
|    40|      1|   5.0|832058959|           6|
|   604|      1|   3.0|832079851|           7|
|   145|      1|   5.0|832105242|           8|
|   130|      1|   3.0|832589610|           9|
|   134|      1|   3.0|832841103|          10|
|   436|      1|   4.0|833529571|          11|
|   314|      1|   3.0|834398280|          12|
|   385|      1|   4.0|834691642|          13|
|    46|      1|   5.0|834787906|          14|
|   584|      1|   5.0|834987643|          15|
|   476|      1|   4.0|835021447|          16|
|   411|      1|   5.0|835532155|          17|
|   541|      1|   3.0|835643027|          18|
|   273|     

# Task 3 — Spark SQL

In [22]:
ratings.createOrReplaceTempView("ratings_table")
movies.createOrReplaceTempView("movies_table")

### 3.1 Requêtes simples SQL

In [24]:
#Show first 20 rows
SELECT * FROM ratings_table LIMIT 20;

SyntaxError: invalid syntax (ipython-input-787027164.py, line 2)

In [25]:
spark.sql("SELECT * FROM ratings_table LIMIT 20").show()

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
|     1|     70|   3.0|964982400|
|     1|    101|   5.0|964980868|
|     1|    110|   4.0|964982176|
|     1|    151|   5.0|964984041|
|     1|    157|   5.0|964984100|
|     1|    163|   5.0|964983650|
|     1|    216|   5.0|964981208|
|     1|    223|   3.0|964980985|
|     1|    231|   5.0|964981179|
|     1|    235|   4.0|964980908|
|     1|    260|   5.0|964981680|
|     1|    296|   3.0|964982967|
|     1|    316|   3.0|964982310|
|     1|    333|   5.0|964981179|
|     1|    349|   4.0|964982563|
+------+-------+------+---------+



In [26]:
spark.sql("SELECT * FROM ratings_table LIMIT 20").show(20, truncate=False)
spark.sql("SELECT COUNT(*) AS total_ratings FROM ratings_table").show()
spark.sql("SELECT COUNT(DISTINCT userId) AS distinct_users FROM ratings_table").show()
spark.sql("SELECT AVG(rating) AS avg_rating FROM ratings_table").show()

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|1     |1      |4.0   |964982703|
|1     |3      |4.0   |964981247|
|1     |6      |4.0   |964982224|
|1     |47     |5.0   |964983815|
|1     |50     |5.0   |964982931|
|1     |70     |3.0   |964982400|
|1     |101    |5.0   |964980868|
|1     |110    |4.0   |964982176|
|1     |151    |5.0   |964984041|
|1     |157    |5.0   |964984100|
|1     |163    |5.0   |964983650|
|1     |216    |5.0   |964981208|
|1     |223    |3.0   |964980985|
|1     |231    |5.0   |964981179|
|1     |235    |4.0   |964980908|
|1     |260    |5.0   |964981680|
|1     |296    |3.0   |964982967|
|1     |316    |3.0   |964982310|
|1     |333    |5.0   |964981179|
|1     |349    |4.0   |964982563|
+------+-------+------+---------+

+-------------+
|total_ratings|
+-------------+
|       100836|
+-------------+

+--------------+
|distinct_users|
+--------------+
|           610|
+--------------+

+---------------

### 3.2 SQL Grouping

In [27]:
# average rating by movie
spark.sql("""
SELECT movieId, AVG(rating) AS avg_rating, COUNT(*) AS cnt
FROM ratings_table
GROUP BY movieId
ORDER BY cnt DESC
LIMIT 20
""").show(truncate=False)

# Best movies with at least 100 ratings
spark.sql("""
SELECT m.movieId, m.title, AVG(r.rating) as avg_rating, COUNT(*) as cnt
FROM ratings_table r
JOIN movies_table m ON r.movieId = m.movieId
GROUP BY m.movieId, m.title
HAVING COUNT(*) >= 100
ORDER BY avg_rating DESC, cnt DESC
LIMIT 20
""").show(truncate=False)

+-------+------------------+---+
|movieId|avg_rating        |cnt|
+-------+------------------+---+
|356    |4.164133738601824 |329|
|318    |4.429022082018927 |317|
|296    |4.197068403908795 |307|
|593    |4.161290322580645 |279|
|2571   |4.192446043165468 |278|
|260    |4.231075697211155 |251|
|480    |3.75              |238|
|110    |4.031645569620253 |237|
|589    |3.970982142857143 |224|
|527    |4.225             |220|
|2959   |4.272935779816514 |218|
|1      |3.9209302325581397|215|
|1196   |4.2156398104265405|211|
|50     |4.237745098039215 |204|
|2858   |4.056372549019608 |204|
|47     |3.9753694581280787|203|
|780    |3.4455445544554455|202|
|150    |3.845771144278607 |201|
|1198   |4.2075            |200|
|4993   |4.106060606060606 |198|
+-------+------------------+---+

+-------+------------------------------------------------------------------------------+------------------+---+
|movieId|title                                                                         |avg_rat

### 3.3 SQL CASE WHEN — buckets

In [28]:
spark.sql("""
SELECT rating,
CASE
  WHEN rating >= 4.0 THEN 'high'
  WHEN rating >= 3.0 THEN 'medium'
  ELSE 'low'
END AS bucket,
COUNT(*) AS cnt
FROM ratings_table
GROUP BY rating
ORDER BY rating DESC
LIMIT 10
""").show(truncate=False)

+------+------+-----+
|rating|bucket|cnt  |
+------+------+-----+
|5.0   |high  |13211|
|4.5   |high  |8551 |
|4.0   |high  |26818|
|3.5   |medium|13136|
|3.0   |medium|20047|
|2.5   |low   |5550 |
|2.0   |low   |7551 |
|1.5   |low   |1791 |
|1.0   |low   |2811 |
|0.5   |low   |1370 |
+------+------+-----+



In [29]:
### 3.4 SQL Window Functions

In [30]:
spark.sql("""
SELECT userId, movieId, rating, timestamp
FROM (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY userId ORDER BY timestamp ASC) as rn
  FROM ratings_table
) t
WHERE rn = 1
LIMIT 20
""").show(truncate=False)

+------+-------+------+----------+
|userId|movieId|rating|timestamp |
+------+-------+------+----------+
|1     |804    |4.0   |964980499 |
|2     |318    |3.0   |1445714835|
|3     |1275   |3.5   |1306463323|
|4     |171    |3.0   |945078428 |
|5     |590    |5.0   |847434747 |
|6     |590    |5.0   |845553109 |
|7     |1784   |0.5   |1106635416|
|8     |150    |4.0   |839463422 |
|9     |41     |3.0   |1044656650|
|10    |92259  |5.0   |1455301553|
|11    |1917   |4.0   |901200037 |
|12    |543    |3.5   |1247263318|
|13    |1639   |4.0   |987456818 |
|14    |592    |2.0   |835440976 |
|15    |172    |1.0   |1299424762|
|16    |2427   |1.5   |1377476573|
|17    |910    |3.5   |1305696226|
|18    |318    |5.0   |1455049328|
|19    |1882   |2.0   |965701107 |
|20    |2804   |5.0   |1054036159|
+------+-------+------+----------+



# Task 4 — Joins & Genre Analytics

### 4.1 Join basique ratings → movies

In [31]:
joined = ratings.join(movies, on="movieId", how="inner")
joined.select("userId","title","rating").show(20, truncate=False)

# Count how many ratings each genre has — requires parsing genres string

+------+-----------------------------------------+------+
|userId|title                                    |rating|
+------+-----------------------------------------+------+
|1     |Toy Story (1995)                         |4.0   |
|1     |Grumpier Old Men (1995)                  |4.0   |
|1     |Heat (1995)                              |4.0   |
|1     |Seven (a.k.a. Se7en) (1995)              |5.0   |
|1     |Usual Suspects, The (1995)               |5.0   |
|1     |From Dusk Till Dawn (1996)               |3.0   |
|1     |Bottle Rocket (1996)                     |5.0   |
|1     |Braveheart (1995)                        |4.0   |
|1     |Rob Roy (1995)                           |5.0   |
|1     |Canadian Bacon (1995)                    |5.0   |
|1     |Desperado (1995)                         |5.0   |
|1     |Billy Madison (1995)                     |5.0   |
|1     |Clerks (1994)                            |3.0   |
|1     |Dumb & Dumber (Dumb and Dumber) (1994)   |5.0   |
|1     |Ed Woo

### 4.2 Parser les genres (split & explode)

In [33]:
from pyspark.sql.functions import col, split, explode

In [34]:
# movies.genres example: "Action|Adventure|Sci-Fi"
movies_genres = movies.withColumn("genre_array", split(col("genres"), "\\|")) \
                      .withColumn("genre", explode(col("genre_array")))
movies_genres.select("movieId","title","genre").show(20, truncate=False)

# join ratings & exploded genres
ratings_with_genre = ratings.join(movies_genres.select("movieId","genre","title"), on="movieId", how="left")

# count ratings per genre
ratings_with_genre.groupBy("genre").agg(
    F.count("*").alias("num_ratings"),
    F.avg("rating").alias("avg_rating")
).orderBy(col("num_ratings").desc()).show(truncate=False)

+-------+----------------------------------+---------+
|movieId|title                             |genre    |
+-------+----------------------------------+---------+
|1      |Toy Story (1995)                  |Adventure|
|1      |Toy Story (1995)                  |Animation|
|1      |Toy Story (1995)                  |Children |
|1      |Toy Story (1995)                  |Comedy   |
|1      |Toy Story (1995)                  |Fantasy  |
|2      |Jumanji (1995)                    |Adventure|
|2      |Jumanji (1995)                    |Children |
|2      |Jumanji (1995)                    |Fantasy  |
|3      |Grumpier Old Men (1995)           |Comedy   |
|3      |Grumpier Old Men (1995)           |Romance  |
|4      |Waiting to Exhale (1995)          |Comedy   |
|4      |Waiting to Exhale (1995)          |Drama    |
|4      |Waiting to Exhale (1995)          |Romance  |
|5      |Father of the Bride Part II (1995)|Comedy   |
|6      |Heat (1995)                       |Action   |
|6      |H

### 4.3 Join + Aggregation

In [35]:
# average rating per genre (already computed above), average per movie title, number of ratings per movie
avg_per_movie = ratings.join(movies, on="movieId").groupBy("movieId","title").agg(
    F.count("*").alias("num_ratings"),
    F.avg("rating").alias("avg_rating")
).orderBy(col("num_ratings").desc())
avg_per_movie.show(20, truncate=False)

# number of ratings per genre per year (extract year from title if title contains year in parentheses)
# Example: "Toy Story (1995)" -> extract 1995
movies_with_year = movies.withColumn("year", F.regexp_extract(col("title"), r".*\((\d{4})\).*", 1))
# join
ratings_movies = ratings.join(movies_genres.select("movieId","genre"), on="movieId", how="left") \
                        .join(movies_with_year.select("movieId","year"), on="movieId", how="left")
# group
ratings_movies.groupBy("genre","year").agg(F.count("*").alias("num_ratings")).orderBy(col("genre"), col("year").desc()).show(50, truncate=False)

+-------+------------------------------------------------------------------------------+-----------+------------------+
|movieId|title                                                                         |num_ratings|avg_rating        |
+-------+------------------------------------------------------------------------------+-----------+------------------+
|356    |Forrest Gump (1994)                                                           |329        |4.164133738601824 |
|318    |Shawshank Redemption, The (1994)                                              |317        |4.429022082018927 |
|296    |Pulp Fiction (1994)                                                           |307        |4.197068403908795 |
|593    |Silence of the Lambs, The (1991)                                              |279        |4.161290322580645 |
|2571   |Matrix, The (1999)                                                            |278        |4.192446043165468 |
|260    |Star Wars: Episode IV - A New H

In [ ]:
### 4.4 Left Anti Join (movies sans ratings)

In [ ]:
movies_with_ratings = movies.join(ratings, on="movieId", how="left_semi")  # movies present in ratings
movies_no_ratings = movies.join(ratings, on="movieId", how="left_anti")    # movies without ratings
print("Movies with no ratings:", movies_no_ratings.count())
movies_no_ratings.show(20, truncate=False)

# Task 5 — MLlib Classification (Predict rating >= 4.0)

### 5.1 Créer label

In [36]:
ratings_ml = ratings.withColumn("label", when(col("rating") >= 4.0, lit(1)).otherwise(lit(0)))
ratings_ml.select("userId","movieId","rating","label").show(10)


+------+-------+------+-----+
|userId|movieId|rating|label|
+------+-------+------+-----+
|     1|      1|   4.0|    1|
|     1|      3|   4.0|    1|
|     1|      6|   4.0|    1|
|     1|     47|   5.0|    1|
|     1|     50|   5.0|    1|
|     1|     70|   3.0|    0|
|     1|    101|   5.0|    1|
|     1|    110|   4.0|    1|
|     1|    151|   5.0|    1|
|     1|    157|   5.0|    1|
+------+-------+------+-----+
only showing top 10 rows


### 5.2 Feature engineering

In [37]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

# user rating counts (feature)
user_counts = ratings.groupBy("userId").agg(F.count("*").alias("user_num_ratings"))
# join
ratings_feats = ratings_ml.join(user_counts, on="userId", how="left")

# add log timestamp
ratings_feats = ratings_feats.withColumn("timestamp_log", log10(col("timestamp") + lit(1)))

# choose features
feature_cols = ["rating","timestamp","timestamp_log","user_num_ratings"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")

data_ml = assembler.transform(ratings_feats).select("raw_features","label").withColumnRenamed("raw_features","features")
data_ml.show(5, truncate=False)

+------------------------------------------+-----+
|features                                  |label|
+------------------------------------------+-----+
|[4.0,9.64982703E8,8.98451952927677,232.0] |1    |
|[4.0,9.64981247E8,8.984518873997418,232.0]|1    |
|[4.0,9.64982224E8,8.984519313700774,232.0]|1    |
|[5.0,9.64983815E8,8.98452002973671,232.0] |1    |
|[5.0,9.64982931E8,8.984519631889107,232.0]|1    |
+------------------------------------------+-----+
only showing top 5 rows


### 5.3 Train/test split

In [38]:
train_df, test_df = data_ml.randomSplit([0.7, 0.3], seed=42)
print("Train size:", train_df.count(), "Test size:", test_df.count())

Train size: 70549 Test size: 30287


### 5.4 Entraîner LogisticRegression

In [39]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=20)
lr_model = lr.fit(train_df)

print("Coefficients:", lr_model.coefficients)
print("Intercept:", lr_model.intercept)

Coefficients: [47.442356697999955,5.1231877564582326e-09,-23.024747883268326,-0.0002595509984819488]
Intercept: 26.181947599140575


### 5.5 Évaluation: AUC, accuracy, precision, recall, confusion matrix

In [40]:
predictions = lr_model.transform(test_df)
predictions.select("features","label","prediction","probability").show(10, truncate=False)

# AUC
bce = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction")
auc = bce.evaluate(predictions)
print("AUC:", auc)

# Accuracy, precision, recall via Multiclass evaluator and manual confusion
mce = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

accuracy = mce.evaluate(predictions, {mce.metricName: "accuracy"})
precision = mce.evaluate(predictions, {mce.metricName: "weightedPrecision"})
recall = mce.evaluate(predictions, {mce.metricName: "weightedRecall"})
print("Accuracy:", accuracy, "Precision:", precision, "Recall:", recall)

# confusion matrix counts
tp = predictions.filter("label = 1 AND prediction = 1").count()
tn = predictions.filter("label = 0 AND prediction = 0").count()
fp = predictions.filter("label = 0 AND prediction = 1").count()
fn = predictions.filter("label = 1 AND prediction = 0").count()
print("TP:",tp,"TN:",tn,"FP:",fp,"FN:",fn)

+--------------------------------------------+-----+----------+-----------+
|features                                    |label|prediction|probability|
+--------------------------------------------+-----+----------+-----------+
|[0.5,1.05302191E9,9.022437407963636,2108.0] |0    |0.0       |[1.0,0.0]  |
|[0.5,1.053021968E9,9.022437431884391,2108.0]|0    |0.0       |[1.0,0.0]  |
|[0.5,1.053069103E9,9.022456871186584,84.0]  |0    |0.0       |[1.0,0.0]  |
|[0.5,1.053198942E9,9.022510414572373,38.0]  |0    |0.0       |[1.0,0.0]  |
|[0.5,1.054037528E9,9.022856074204281,242.0] |0    |0.0       |[1.0,0.0]  |
|[0.5,1.054037842E9,9.022856203581503,242.0] |0    |0.0       |[1.0,0.0]  |
|[0.5,1.054037857E9,9.022856209761942,242.0] |0    |0.0       |[1.0,0.0]  |
|[0.5,1.054147297E9,9.022901299907849,242.0] |0    |0.0       |[1.0,0.0]  |
|[0.5,1.054779634E9,9.023161736126607,977.0] |0    |0.0       |[1.0,0.0]  |
|[0.5,1.054779861E9,9.023161829591473,977.0] |0    |0.0       |[1.0,0.0]  |
+-----------

### 5.6 Améliorations: Log timestamp, DecisionTree, RandomForest (exemples)

In [41]:
# Try DecisionTree
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier

dt = DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=5)
dt_model = dt.fit(train_df)
dt_preds = dt_model.transform(test_df)
print("DT AUC:", BinaryClassificationEvaluator(labelCol="label").evaluate(dt_preds))

# RandomForest
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50, maxDepth=7)
rf_model = rf.fit(train_df)
rf_preds = rf_model.transform(test_df)
print("RF AUC:", BinaryClassificationEvaluator(labelCol="label").evaluate(rf_preds))

# Compare accuracy
for name, preds in [("LR",predictions),("DT",dt_preds),("RF",rf_preds)]:
    acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy").evaluate(preds)
    print(name, "accuracy:", acc)

DT AUC: 1.0
RF AUC: 1.0
LR accuracy: 1.0
DT accuracy: 1.0
RF accuracy: 1.0


In [42]:
# Task 6 — Performance & Execution

In [43]:
# 6.1 Nombre de partitions

In [44]:
print("ratings partitions:", ratings.rdd.getNumPartitions())
print("movies partitions:", movies.rdd.getNumPartitions())

ratings partitions: 1
movies partitions: 1


In [45]:
### 6.2 Repartition vs Coalesce (démo)

In [46]:
# Repartition (shuffle) -> augmente / change les partitions
ratings_r8 = ratings.repartition(8)
print("After repartition(8):", ratings_r8.rdd.getNumPartitions())

# Coalesce (pas de shuffle si reducePartitions=True) -> réduire partitions sans shuffle généralement
ratings_c2 = ratings_r8.coalesce(2)
print("After coalesce(2):", ratings_c2.rdd.getNumPartitions())

After repartition(8): 8
After coalesce(2): 2


In [47]:
### 6.3 Cache (mesurer temps)

In [48]:
import time
ratings_cached = ratings.repartition(4).cache()

t0 = time.time()
cnt = ratings_cached.count()   # materialize
t1 = time.time()
# heavy aggregation twice
t2 = time.time()
ratings_cached.groupBy("movieId").agg(F.avg("rating")).count()
t3 = time.time()
ratings_cached.groupBy("movieId").agg(F.avg("rating")).count()
t4 = time.time()

print("initial count time:", t1-t0)
print("first agg time:", t3-t2)
print("second agg time (cached):", t4-t3)

initial count time: 1.3680038452148438
first agg time: 0.7200207710266113
second agg time (cached): 0.5836842060089111


In [49]:
### 6.4 explain(True) sur quelques opérations

In [50]:
# join explain
jdf = ratings.join(movies, on="movieId")
jdf.explain(True)

# groupBy explain
gdf = ratings.groupBy("movieId").agg(F.avg("rating").alias("avg_rating"))
gdf.explain(True)

# window explain (partition by movieId)
w = Window.partitionBy("movieId").orderBy(col("timestamp"))
win_df = ratings.withColumn("rn", F.row_number().over(w))
win_df.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [movieId])
:- Relation [userId#17,movieId#18,rating#19,timestamp#20] csv
+- Relation [movieId#38,title#39,genres#40] csv

== Analyzed Logical Plan ==
movieId: int, userId: int, rating: double, timestamp: int, title: string, genres: string
Project [movieId#18, userId#17, rating#19, timestamp#20, title#39, genres#40]
+- Join Inner, (movieId#18 = movieId#38)
   :- Relation [userId#17,movieId#18,rating#19,timestamp#20] csv
   +- Relation [movieId#38,title#39,genres#40] csv

== Optimized Logical Plan ==
Project [movieId#18, userId#17, rating#19, timestamp#20, title#39, genres#40]
+- Join Inner, (movieId#18 = movieId#38)
   :- Filter isnotnull(movieId#18)
   :  +- Relation [userId#17,movieId#18,rating#19,timestamp#20] csv
   +- Filter isnotnull(movieId#38)
      +- Relation [movieId#38,title#39,genres#40] csv

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [movieId#18, userId#17, rating#19, timestamp#20, title#39, genres#40